In [ ]:
# ===================================================================================
# MIGRATION PRE-FLIGHT CHECK — WEB SCENES (3D Maps)
# - Scans Web Scenes for dependencies (operational layers, ground layers, basemap)
# - Checks if enterprise Scene Services / Feature Services exist on Target
# - Uses ledger + SourceID tag fallback for verification
# - Reports READY vs BLOCKED scenes and lists missing service IDs
#
# READ ONLY: Does not create or modify any items.
# ===================================================================================

import pandas as pd
import os
import json
import urllib3
import requests
from arcgis.gis import GIS

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
requests.packages.urllib3.disable_warnings()

# =============================================================================
# --- CONFIGURATION (from shared config) ---------------------------------------
# =============================================================================
from migration_config import *

# --- ORCHESTRATOR SIDECAR LOADER ---
_sidecar_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "_sidecar_preflight_webscenes.json")
if os.path.exists(_sidecar_path):
    import json as _json
    WEB_SCENE_IDS_TO_CHECK = _json.load(open(_sidecar_path))["ids"]
    print(f"[Orchestrator] Loaded {len(WEB_SCENE_IDS_TO_CHECK)} Web Scene IDs from sidecar.")
else:
    WEB_SCENE_IDS_TO_CHECK = [
        # Example Source ID's
        # "abc123def456...",
    ]

# =============================================================================
# --- LOAD HISTORY + LEDGER ----------------------------------------------------
# =============================================================================
ALREADY_MIGRATED_IDS = set()
MIGRATION_LEDGER = {}  # (SourceID:str, LayerIndex:int) -> TargetID:str

def load_history_and_ledger():
    global ALREADY_MIGRATED_IDS, MIGRATION_LEDGER
    if not os.path.exists(LOG_FILE):
        print(f"ℹ No history file found at: {LOG_FILE}")
        return
    try:
        df = pd.read_csv(LOG_FILE)
        if "SourceID" in df.columns:
            ALREADY_MIGRATED_IDS = set(df["SourceID"].astype(str).str.strip())
        required = {"SourceID", "LayerIndex", "TargetID"}
        if required.issubset(set(df.columns)):
            for _, row in df.iterrows():
                sid = str(row.get("SourceID", "")).strip()
                tid = str(row.get("TargetID", "")).strip()
                idx = row.get("LayerIndex", "")
                if not sid or not tid: continue
                try: idx_int = int(idx)
                except: continue
                MIGRATION_LEDGER[(sid, idx_int)] = tid
        print(f"Loaded history: {len(ALREADY_MIGRATED_IDS)} SourceIDs")
        print(f"Loaded ledger entries: {len(MIGRATION_LEDGER)} (SourceID,LayerIndex)->TargetID")
    except Exception as e:
        print(f"⚠ Could not read history/ledger: {e}")

# =============================================================================
# --- CONNECT ------------------------------------------------------------------
# =============================================================================
print("Connecting for Web Scene Scan...")
load_history_and_ledger()

source_gis = GIS(url=SOURCE_URL, token=SOURCE_TOKEN, verify_cert=False, referer=SOURCE_URL)
target_gis = GIS(url=TARGET_URL, token=TARGET_TOKEN, verify_cert=False, referer=TARGET_URL)

print(f"   > Source: {source_gis.url}")
print(f"   > Target User: {target_gis.users.me.username}")

# =============================================================================
# --- HELPERS ------------------------------------------------------------------
# =============================================================================
def iter_layers_recursive(layer_list):
    """Yield every layer dict including nested group layers."""
    if not isinstance(layer_list, list): return
    for lyr in layer_list:
        if isinstance(lyr, dict):
            yield lyr
            if "layers" in lyr and isinstance(lyr["layers"], list):
                yield from iter_layers_recursive(lyr["layers"])

def classify_layer(layer):
    """
    Returns: (class, reason)
    class in {"enterprise", "agol", "external", "no_url"}
    """
    url = (layer.get("url", "") or "").lower()
    style_url = (layer.get("styleUrl", "") or "").lower()

    # Check styleUrl first (Vector Tile Layers)
    if style_url:
        if "arcgis.com" in style_url or "arcgisonline.com" in style_url:
            return "agol", "VectorTileLayer styleUrl points to ArcGIS Online."
        return "external", "VectorTileLayer styleUrl points to external host."

    if not url:
        return "no_url", "Layer has no 'url' (and no styleUrl) — manual review needed."

    if "arcgisonline.com" in url or "services.arcgisonline.com" in url or "arcgis.com" in url:
        return "agol", "ArcGIS Online / Living Atlas reference."

    if ENTERPRISE_HOST in url:
        return "enterprise", "Enterprise-hosted service."

    return "external", "Non-enterprise external host."

def resolve_source_service_id(layer):
    """Resolve Source enterprise service item ID from layer JSON."""
    if "itemId" in layer and layer["itemId"]:
        return layer["itemId"]
    url = layer.get("url", "") or ""
    if url and ENTERPRISE_HOST in url:
        for server_type in ["/SceneServer", "/FeatureServer", "/MapServer"]:
            if server_type in url:
                base = url.split(server_type)[0] + server_type
                try:
                    res = source_gis.content.search(f'url:"{base}"', max_items=1)
                    if res: return res[0].id
                except: pass
                break
    return None

def target_exists_for_dependency(source_service_id):
    """Check ledger + tag fallback for any matching target item."""
    sid = str(source_service_id).strip()

    # Ledger check (any layer index)
    for (ledger_sid, _), t_id in MIGRATION_LEDGER.items():
        if ledger_sid == sid:
            try:
                if target_gis.content.get(t_id): return True
            except: pass

    # Tag fallback
    try:
        res = target_gis.content.search(f'tags:"SourceID:{sid}"', max_items=1)
        if res: return True
    except: pass

    return False

# =============================================================================
# --- MAIN SCAN ----------------------------------------------------------------
# =============================================================================
print(f"\nScanning {len(WEB_SCENE_IDS_TO_CHECK)} Web Scenes for dependencies...\n")

scenes_ready = []
scenes_blocked = []
missing_enterprise_services = set()
blocked_details = []
external_notes = []

for scene_id in WEB_SCENE_IDS_TO_CHECK:
    try:
        web_scene = source_gis.content.get(scene_id)
        if not web_scene:
            print(f"❌ Scene {scene_id} not found.")
            scenes_blocked.append(f"{scene_id} (not found)")
            continue

        data = web_scene.get_data()
        if not isinstance(data, dict):
            print(f"⚠ Scene {scene_id} returned non-dict data; skipping.")
            scenes_blocked.append(f"{web_scene.title} (bad data)")
            continue

        # Collect all layer containers
        containers = []
        if "operationalLayers" in data:
            containers.append(("operationalLayers", data["operationalLayers"]))
        if "baseMap" in data and isinstance(data.get("baseMap"), dict):
            if "baseMapLayers" in data["baseMap"]:
                containers.append(("baseMapLayers", data["baseMap"]["baseMapLayers"]))
        if "ground" in data and isinstance(data.get("ground"), dict):
            if "layers" in data["ground"]:
                containers.append(("groundLayers", data["ground"]["layers"]))

        scene_blockers = []

        for container_name, layer_list in containers:
            for layer in iter_layers_recursive(layer_list):
                title = layer.get("title", "(no title)")
                layer_type = layer.get("layerType", layer.get("type", ""))
                url = layer.get("url", "") or ""

                cls, cls_reason = classify_layer(layer)

                if cls in ("agol", "external"):
                    note = {
                        "SceneTitle": web_scene.title,
                        "SceneID": scene_id,
                        "Container": container_name,
                        "LayerTitle": title,
                        "LayerType": layer_type,
                        "URL": url or layer.get("styleUrl", ""),
                        "Class": cls,
                        "Reason": cls_reason,
                    }
                    if BLOCK_EXTERNAL_LAYERS:
                        scene_blockers.append({
                            "SceneTitle": web_scene.title, "SceneID": scene_id,
                            "Container": container_name, "LayerTitle": title,
                            "SourceServiceID": "EXTERNAL",
                            "Reason": f"External dependency blocked by policy. {cls_reason}",
                        })
                    else:
                        external_notes.append(note)
                    continue

                if cls == "no_url":
                    scene_blockers.append({
                        "SceneTitle": web_scene.title, "SceneID": scene_id,
                        "Container": container_name, "LayerTitle": title,
                        "SourceServiceID": "UNKNOWN",
                        "Reason": "Layer has no URL and no styleUrl; manual review needed.",
                    })
                    continue

                # Enterprise dependency
                src_sid = resolve_source_service_id(layer)
                if not src_sid:
                    scene_blockers.append({
                        "SceneTitle": web_scene.title, "SceneID": scene_id,
                        "Container": container_name, "LayerTitle": title,
                        "SourceServiceID": "UNKNOWN",
                        "Reason": "Enterprise URL but could not resolve source itemId.",
                    })
                    continue

                if not target_exists_for_dependency(src_sid):
                    scene_blockers.append({
                        "SceneTitle": web_scene.title, "SceneID": scene_id,
                        "Container": container_name, "LayerTitle": title,
                        "SourceServiceID": str(src_sid),
                        "Reason": "Enterprise dependency missing on target.",
                    })
                    missing_enterprise_services.add(str(src_sid))

        if scene_blockers:
            print(f"🔴 BLOCKED: {web_scene.title}")
            for b in scene_blockers:
                print(f"   - {b['LayerTitle']} | SourceID={b['SourceServiceID']} | {b['Reason']}")
            scenes_blocked.append(web_scene.title)
            blocked_details.extend(scene_blockers)
        else:
            print(f"🟢 READY:   {web_scene.title}")
            scenes_ready.append(web_scene.title)

    except Exception as e:
        print(f"❌ Error scanning {scene_id}: {e}")
        scenes_blocked.append(f"{scene_id} (error)")

# =============================================================================
# --- FINAL REPORT -------------------------------------------------------------
# =============================================================================
print("\n" + "=" * 60)
print("              WEB SCENE PRE-FLIGHT REPORT")
print("=" * 60)
print(f"✅ Scenes Ready to Migrate:        {len(scenes_ready)}")
print(f"🔴 Scenes Blocked (needs action):  {len(scenes_blocked)}")
print(f"⚠ Missing Enterprise Service IDs: {len(missing_enterprise_services)}")
print(f"ℹ External/AGOL layers noted:     {len(external_notes)}")
print("-" * 60)

if missing_enterprise_services:
    print("\n⚠️  MISSING ENTERPRISE SERVICES (migrate these first):\n")
    print("SERVICE_IDS = [")
    for sid in sorted(missing_enterprise_services):
        try:
            item = source_gis.content.get(sid)
            name = item.title if item else "Unknown"
            print(f'    "{sid}",  # {name}')
        except:
            print(f'    "{sid}",')
    print("]")

if external_notes and not BLOCK_EXTERNAL_LAYERS:
    print(f"\n🌐 EXTERNAL / AGOL LAYERS ({len(external_notes)} total, usually OK to keep):")
    for r in external_notes[:50]:
        print(f"   - {r['SceneTitle']}: {r['LayerTitle']} ({r['Class']})")
    if len(external_notes) > 50:
        print(f"   ... (+{len(external_notes) - 50} more)")

if scenes_blocked:
    print("\n🛑 Some scenes are blocked. Resolve blockers above before migrating.")
else:
    print("\n🚀 All systems go! No blockers detected for the scanned scenes.")
print("=" * 60)

# --- ORCHESTRATOR OUTPUT ---
import json as _json
_output_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "_output_preflight_webscenes.json")
with open(_output_path, "w", encoding="utf-8") as _f:
    _json.dump({"missing_ids": list(missing_enterprise_services)}, _f, indent=2)
print(f"[Orchestrator] Wrote {len(missing_enterprise_services)} missing service IDs to {os.path.basename(_output_path)}")